In [ ]:
# !pip install -q transformers sentence-transformers
!pip install -q torch-geometric
# !pip install -q spacy
# !python -m spacy download en_core_web_sm

In [ ]:
!pip install -q torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.1 MB/s eta 0:00:00


In [ ]:
import os
import spacy
import torch
import random
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
from PIL import Image
from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoProcessor,
    CLIPVisionModel
)
from torch.cuda.amp import (
    autocast,
    GradScaler
)

from torch.utils.data import (
    Dataset,
    DataLoader
)

from torch_geometric.data import (
    Data,
    Batch
)

from torch_geometric.nn import (
    GATv2Conv,
    global_mean_pool
)

from sklearn.metrics import (
    accuracy_score,
    f1_score
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import re

In [ ]:
class CFG:

    TRAIN_CSV="/content/train.csv"

    DEV_CSV="/content/dev.csv"

    GRAPH_CACHE="/content/graph_cache.pt"

    DEVICE="cuda" if torch.cuda.is_available() else "cpu"

    TEXT_MODEL="sentence-transformers/all-MiniLM-L6-v2"

    IMAGE_MODEL="openai/clip-vit-base-patch32"

    MAX_LEN=64

    BATCH_SIZE=8

    EPOCHS=5

    LR=2e-5

    GRAPH_DIM=128

    HIDDEN_DIM=384

    DROPOUT=0.2


In [ ]:
def seed_everything(seed=42):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

seed_everything()

In [ ]:
nlp=spacy.load("en_core_web_sm")

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(
    CFG.TEXT_MODEL
)
image_processor=AutoProcessor.from_pretrained(
    CFG.IMAGE_MODEL
)
text_encoder=AutoModel.from_pretrained(
    CFG.TEXT_MODEL
).to(CFG.DEVICE)

image_encoder=CLIPVisionModel.from_pretrained(
    CFG.IMAGE_MODEL
).to(CFG.DEVICE)

for p in text_encoder.parameters():
    p.requires_grad=False

for p in image_encoder.parameters():
    p.requires_grad=False

text_encoder.eval()

image_encoder.eval()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.final_layer_norm.bias                             | UNEXPECTED |  | 
text_model.final_layer_norm.weight                           | UNEXPECTED |  | 
logit_scale                                                  | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weigh

CLIPVisionModel(
  (vision_model): CLIPVisionTransformer(
    (embeddings): CLIPVisionEmbeddings(
      (patch_embedding): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
      (position_embedding): Embedding(50, 768)
    )
    (pre_layrnorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
        

In [ ]:
@torch.no_grad()
def build_graph(text):

    doc=nlp(text)

    tokens=[]

    for token in doc:

        if token.is_stop:
            continue

        if not token.is_alpha:
            continue

        tokens.append(token.text.lower())

    if len(tokens)==0:
        tokens=["empty"]

    unique_tokens=list(dict.fromkeys(tokens))

    node_map={
        tok:i for i,tok in enumerate(unique_tokens)
    }

    encoded=tokenizer(
        unique_tokens,
        padding=True,
        truncation=True,
        max_length=8,
        return_tensors="pt"
    ).to(CFG.DEVICE)

    outputs=text_encoder(**encoded)

    node_features=outputs.last_hidden_state[:,0,:].cpu()

    edges=[]

    for token in doc:

        if token.text.lower() not in node_map:
            continue

        if token.head.text.lower() not in node_map:
            continue

        src=node_map[token.head.text.lower()]

        dst=node_map[token.text.lower()]

        edges.append([src,dst])

    if len(edges)==0:
        edges=[[0,0]]

    edge_index=torch.tensor(
        edges,
        dtype=torch.long
    ).t().contiguous()

    graph=Data(
        x=node_features,
        edge_index=edge_index
    )

    return graph

In [ ]:
def build_graph_cache(csv_path):

    df=pd.read_csv(csv_path)

    graphs=[]

    for text in tqdm(df["transcription"].astype(str).tolist()):

        graphs.append(build_graph(text))

    return graphs

if not os.path.exists(CFG.GRAPH_CACHE):

    train_graphs=build_graph_cache(
        CFG.TRAIN_CSV
    )

    dev_graphs=build_graph_cache(
        CFG.DEV_CSV
    )

    torch.save({
        "train":train_graphs,
        "dev":dev_graphs
    },CFG.GRAPH_CACHE)

graph_cache=torch.load(
    CFG.GRAPH_CACHE,
    weights_only=False
)

In [ ]:
class MemeDataset(Dataset):

    def __init__(self,csv_path,graphs):

        self.df=pd.read_csv(csv_path)

        self.graphs=graphs

    def __len__(self):

        return len(self.df)

    def __getitem__(self,idx):

        row=self.df.iloc[idx]

        image=Image.open(
            row["image_path"]
        ).convert("RGB")

        text=str(
            row["transcription"]
        )

        graph=self.graphs[idx]

        label=torch.tensor(
            row["label"],
            dtype=torch.float32
        )

        return {
            "image":image,
            "text":text,
            "graph":graph,
            "label":label
        }

In [ ]:
def collate_fn(batch):

    images=[
        x["image"] for x in batch
    ]

    texts=[
        x["text"] for x in batch
    ]

    graphs=Batch.from_data_list(
        [x["graph"] for x in batch]
    )

    labels=torch.stack([
        x["label"] for x in batch
    ])

    return {
        "images":images,
        "texts":texts,
        "graphs":graphs,
        "labels":labels
    }

In [ ]:

class GraphEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.gat1=GATv2Conv(
            384,
            CFG.GRAPH_DIM,
            heads=2,
            dropout=CFG.DROPOUT
        )

        self.gat2=GATv2Conv(
            CFG.GRAPH_DIM*2,
            CFG.GRAPH_DIM,
            heads=1,
            concat=False,
            dropout=CFG.DROPOUT
        )

        self.proj=nn.Linear(
            CFG.GRAPH_DIM,
            384
        )

    def forward(self,graph):

        x=graph.x.to(CFG.DEVICE)

        edge_index=graph.edge_index.to(
            CFG.DEVICE
        )

        batch=graph.batch.to(
            CFG.DEVICE
        )

        x=self.gat1(
            x,
            edge_index
        )

        x=F.elu(x)

        x=self.gat2(
            x,
            edge_index
        )

        x=global_mean_pool(
            x,
            batch
        )

        x=self.proj(x)

        return x

In [ ]:
class Fusion(nn.Module):

    def __init__(self):

        super().__init__()

        self.attn=nn.MultiheadAttention(
            embed_dim=384,
            num_heads=4,
            batch_first=True
        )

        self.fc=nn.Sequential(

            nn.Linear(384*3,384),

            nn.LayerNorm(384),

            nn.GELU(),

            nn.Dropout(CFG.DROPOUT)
        )

    def forward(
        self,
        text_emb,
        image_emb,
        graph_emb
    ):

        print("TEXT:",text_emb.shape)
        print("IMAGE:",image_emb.shape)
        print("GRAPH:",graph_emb.shape)

        if image_emb.shape[-1]!=384:
            raise ValueError(
                f"Image dim wrong: {image_emb.shape}"
            )

        if text_emb.shape[-1]!=384:
            raise ValueError(
                f"Text dim wrong: {text_emb.shape}"
            )

        if graph_emb.shape[-1]!=384:
            raise ValueError(
                f"Graph dim wrong: {graph_emb.shape}"
            )

        text_emb=text_emb.unsqueeze(1)

        image_emb=image_emb.unsqueeze(1)

        attn_out,_=self.attn(
            text_emb,
            image_emb,
            image_emb
        )

        fused=torch.cat([
            attn_out.squeeze(1),
            image_emb.squeeze(1),
            graph_emb
        ],dim=1)

        fused=self.fc(fused)

        return fused

In [ ]:
class MemeModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.graph_encoder=GraphEncoder()

        self.fusion=Fusion()

        self.image_proj=nn.Linear(
            768,
            384
        )

        self.classifier=nn.Sequential(

            nn.Linear(384,256),

            nn.LayerNorm(256),

            nn.GELU(),

            nn.Dropout(CFG.DROPOUT),

            nn.Linear(256,1)
        )

    @torch.no_grad()
    def encode_text(self,texts):

        encoded=tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=CFG.MAX_LEN,
            return_tensors="pt"
        ).to(CFG.DEVICE)

        outputs=text_encoder(**encoded)

        emb=outputs.last_hidden_state[:,0,:]

        return emb

    @torch.no_grad()
    def encode_image(self,images):

        encoded=image_processor(
            images=images,
            return_tensors="pt"
        ).to(CFG.DEVICE)

        outputs=image_encoder(**encoded)

        emb=outputs.pooler_output

        emb=self.image_proj(emb)

        return emb

    def forward(self,batch):

        text_emb=self.encode_text(
            batch["texts"]
        )

        image_emb=self.encode_image(
            batch["images"]
        )

        graph_emb=self.graph_encoder(
            batch["graphs"]
        )

        fused=self.fusion(
            text_emb,
            image_emb,
            graph_emb
        )

        logits=self.classifier(
            fused
        )

        return logits.squeeze(1)

In [ ]:
train_dataset=MemeDataset(
    CFG.TRAIN_CSV,
    graph_cache["train"]
)

In [ ]:
dev_dataset=MemeDataset(
    CFG.DEV_CSV,
    graph_cache["dev"]
)

In [ ]:
train_loader=DataLoader(
    train_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    pin_memory=True
)

dev_loader=DataLoader(
    dev_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    pin_memory=True
)

In [ ]:
model=MemeModel().to(CFG.DEVICE)

In [ ]:
criterion=nn.BCEWithLogitsLoss()

optimizer=torch.optim.AdamW(
    model.parameters(),
    lr=CFG.LR
)

In [ ]:
for epoch in range(CFG.EPOCHS):

    model.train()

    train_losses=[]

    for batch in tqdm(train_loader):

        labels=batch["labels"].to(
            CFG.DEVICE
        )

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):

            logits=model(batch)

            loss=criterion(
                logits,
                labels
            )

        loss.backward()

        optimizer.step()

        train_losses.append(
            loss.item()
        )

    print(
        f"Epoch {epoch+1} Loss:",
        np.mean(train_losses)
    )

  0%|          | 1/1175 [00:03<1:12:29,  3.70s/it]

TEXT: torch.Size([8, 384])
IMAGE: torch.Size([8, 384])
GRAPH: torch.Size([8, 384])


  0%|          | 2/1175 [00:07<1:10:40,  3.62s/it]

TEXT: torch.Size([8, 384])
IMAGE: torch.Size([8, 384])
GRAPH: torch.Size([8, 384])


  0%|          | 3/1175 [00:10<1:11:43,  3.67s/it]

...

TEXT: torch.Size([8, 384])
IMAGE: torch.Size([8, 384])
GRAPH: torch.Size([8, 384])
TEXT: torch.Size([8, 384])
IMAGE: torch.Size([8, 384])
GRAPH: torch.Size([8, 384])


100%|█████████▉| 1173/1175 [03:11<00:00,  6.84it/s]

TEXT: torch.Size([8, 384])
IMAGE: torch.Size([8, 384])
GRAPH: torch.Size([8, 384])
TEXT: torch.Size([8, 384])
IMAGE: torch.Size([8, 384])
GRAPH: torch.Size([8, 384])


100%|██████████| 1175/1175 [03:11<00:00,  6.12it/s]

TEXT: torch.Size([8, 384])
IMAGE: torch.Size([8, 384])
GRAPH: torch.Size([8, 384])
TEXT: torch.Size([3, 384])
IMAGE: torch.Size([3, 384])
GRAPH: torch.Size([3, 384])
Epoch 5 Loss: 0.4034218884909407


In [ ]:
# ============================================================
# EVALUATION
# ============================================================

model.eval()

all_preds=[]

all_labels=[]

with torch.no_grad():

    for batch in tqdm(dev_loader):

        labels=batch["labels"].to(
            CFG.DEVICE
        )

        with torch.amp.autocast("cuda"):

            logits=model(batch)

        probs=torch.sigmoid(logits)

        preds=(probs>0.5).long()

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

acc=accuracy_score(
    all_labels,
    all_preds
)

f1=f1_score(
    all_labels,
    all_preds
)

print("Accuracy:",acc)

print("F1:",f1)

# ============================================================
# SAVE
# ============================================================

torch.save(
    model.state_dict(),
    "lightweight_multimodal_model.pt"
)

print("Model Saved")

  1%|▏         | 1/75 [00:03<04:40,  3.79s/it]

TEXT: torch.Size([8, 384])
IMAGE: torch.Size([8, 384])
GRAPH: torch.Size([8, 384])


  3%|▎         | 2/75 [00:07<04:18,  3.54s/it]

TEXT: torch.Size([8, 384])
IMAGE: torch.Size([8, 384])
GRAPH: torch.Size([8, 384])


  4%|▍         | 3/75 [00:10<04:07,  3.44s/it]

...

TEXT: torch.Size([8, 384])
IMAGE: torch.Size([8, 384])
GRAPH: torch.Size([8, 384])


 99%|█████████▊| 74/75 [04:12<00:03,  3.24s/it]

TEXT: torch.Size([8, 384])
IMAGE: torch.Size([8, 384])
GRAPH: torch.Size([8, 384])


100%|██████████| 75/75 [04:16<00:00,  3.42s/it]

TEXT: torch.Size([8, 384])
IMAGE: torch.Size([8, 384])
GRAPH: torch.Size([8, 384])
Accuracy: 0.785
F1: 0.5930599369085173
Model Saved


In [ ]:
TEST_TEXT_CSV = "/content/test.csv"
TEST_IMG_DIR  = "/content/drive/MyDrive/test_eng"

In [ ]:
txt_df = pd.read_csv(TEST_TEXT_CSV)

# expected columns: image_id, transcriptions
txt_df = txt_df[["image_id", "transcriptions"]].copy()
txt_df.columns = ["image_id", "transcription"]

txt_df["image_id"] = pd.to_numeric(txt_df["image_id"], errors="coerce")
txt_df = txt_df.dropna(subset=["image_id"]).reset_index(drop=True)
txt_df["image_id"] = txt_df["image_id"].astype(int)

In [ ]:
rows = []
valid_ext = (".jpg", ".jpeg", ".png", ".webp", ".bmp", ".jfif")

for root, dirs, files in os.walk(TEST_IMG_DIR):
    for fname in files:
        if fname.lower().endswith(valid_ext):
            path = os.path.join(root, fname)
            stem = os.path.splitext(fname)[0]

            digits = re.sub(r"\D", "", stem)
            if digits == "":
                continue

            img_id = int(digits)

            rows.append({
                "image_id": img_id,
                "image_path": path
            })

img_df = pd.DataFrame(rows)
img_df = img_df.drop_duplicates(subset="image_id", keep="first")


In [ ]:
FUSED_TEST_CSV = "fused_test.csv"

In [ ]:
test_df = txt_df.merge(img_df, on="image_id", how="inner")
test_df = test_df[["image_id", "image_path", "transcription"]]

print("Matched rows:", len(test_df))
print(test_df.head())

test_df.to_csv(FUSED_TEST_CSV, index=False)
print("Saved fused test:", FUSED_TEST_CSV)

Matched rows: 1000
   image_id                                 image_path  \
0     15236  /content/drive/MyDrive/test_eng/15236.jpg   
1     15805  /content/drive/MyDrive/test_eng/15805.jpg   
2     16254  /content/drive/MyDrive/test_eng/16254.jpg   
3     16191  /content/drive/MyDrive/test_eng/16191.jpg   
4     15952  /content/drive/MyDrive/test_eng/15952.jpg   

                                       transcription  
0  FACEBOOK SINGLES GROUPS BELIKE WHEN A NEW WOMA...  
1    SO, IF YOU'RE A FEMINIST HOW CAN YOU EAT DAIRY?  
2         WHEN A CUTE GIRL LEFT YOUR MESSAGE ON SEEN  
3  Photographing something you want to show every...  
4  HEY BABE CAN YOU MAKE ME A SANDWICH? Hey babe ...  
Saved fused test: fused_test.csv


In [ ]:
ddd=pd.read_csv('/content/test.csv')

In [ ]:
ddd.head()

,image_id,image_path,transcription
0,15236,/content/drive/MyDrive/test_eng/15236.jpg,FACEBOOK SINGLES GROUPS BELIKE WHEN A NEW WOMA...
1,15805,/content/drive/MyDrive/test_eng/15805.jpg,"SO, IF YOU'RE A FEMINIST HOW CAN YOU EAT DAIRY?"
2,16254,/content/drive/MyDrive/test_eng/16254.jpg,WHEN A CUTE GIRL LEFT YOUR MESSAGE ON SEEN
3,16191,/content/drive/MyDrive/test_eng/16191.jpg,Photographing something you want to show every...
4,15952,/content/drive/MyDrive/test_eng/15952.jpg,HEY BABE CAN YOU MAKE ME A SANDWICH? Hey babe ...


In [ ]:
ddd.shape()

TypeError: 'tuple' object is not callable

In [ ]:
# ============================================================
# TEST DATASET
# ============================================================

class TestDataset(Dataset):

    def __init__(self,csv_path,graphs):

        self.df=pd.read_csv(csv_path)

        self.graphs=graphs

    def __len__(self):

        return len(self.df)

    def __getitem__(self,idx):

        row=self.df.iloc[idx]

        image=Image.open(
            row["image_path"]
        ).convert("RGB")

        text=str(
            row["transcription"]
        )

        graph=self.graphs[idx]

        return {
            "image":image,
            "text":text,
            "graph":graph
        }

# ============================================================
# TEST COLLATE
# ============================================================

def test_collate_fn(batch):

    images=[
        x["image"] for x in batch
    ]

    texts=[
        x["text"] for x in batch
    ]

    graphs=Batch.from_data_list(
        [x["graph"] for x in batch]
    )

    return {
        "images":images,
        "texts":texts,
        "graphs":graphs
    }

# ============================================================
# TEST GRAPH CACHE
# ============================================================

TEST_CSV="/content/test.csv"

TEST_GRAPH_CACHE="/content/test_graphs.pt"

if not os.path.exists(TEST_GRAPH_CACHE):

    test_graphs=build_graph_cache(
        TEST_CSV
    )

    torch.save(
        test_graphs,
        TEST_GRAPH_CACHE
    )

test_graphs=torch.load(
    TEST_GRAPH_CACHE,
    weights_only=False
)

# ============================================================
# TEST LOADER
# ============================================================

test_dataset=TestDataset(
    TEST_CSV,
    test_graphs
)

test_loader=DataLoader(
    test_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    collate_fn=test_collate_fn,
    pin_memory=True
)

# ============================================================
# TEST PREDICTION
# ============================================================

model.eval()

all_probs=[]

all_preds=[]

with torch.no_grad():

    for batch in tqdm(test_loader):

        with torch.amp.autocast("cuda"):

            logits=model(batch)

        probs=torch.sigmoid(logits)

        preds=(probs>0.5).long()

        all_probs.extend(
            probs.cpu().numpy()
        )

        all_preds.extend(
            preds.cpu().numpy()
        )

# ============================================================
# SAVE PREDICTIONS
# ============================================================

test_df=pd.read_csv(TEST_CSV)

test_df["prediction"]=all_preds

test_df["probability"]=all_probs

test_df.to_csv(
    "test_predictions.csv",
    index=False
)

print(test_df.head())

print("Test Predictions Saved")

  1%|          | 1/125 [00:04<08:23,  4.06s/it]

TEXT: torch.Size([8, 384])
IMAGE: torch.Size([8, 384])
GRAPH: torch.Size([8, 384])


  2%|▏         | 2/125 [00:07<07:32,  3.67s/it]

TEXT: torch.Size([8, 384])
IMAGE: torch.Size([8, 384])
GRAPH: torch.Size([8, 384])


  2%|▏         | 3/125 [00:11<07:26,  3.66s/it]

...

 99%|█████████▉| 124/125 [07:09<00:03,  3.41s/it]

TEXT: torch.Size([8, 384])
IMAGE: torch.Size([8, 384])
GRAPH: torch.Size([8, 384])


100%|██████████| 125/125 [07:12<00:00,  3.46s/it]

TEXT: torch.Size([8, 384])
IMAGE: torch.Size([8, 384])
GRAPH: torch.Size([8, 384])
   image_id                                 image_path  \
0     15236  /content/drive/MyDrive/test_eng/15236.jpg   
1     15805  /content/drive/MyDrive/test_eng/15805.jpg   
2     16254  /content/drive/MyDrive/test_eng/16254.jpg   
3     16191  /content/drive/MyDrive/test_eng/16191.jpg   
4     15952  /content/drive/MyDrive/test_eng/15952.jpg   

                                       transcription  prediction  probability  
0  FACEBOOK SINGLES GROUPS BELIKE WHEN A NEW WOMA...           0     0.130371  
1    SO, IF YOU'RE A FEMINIST HOW CAN YOU EAT DAIRY?           0     0.289795  
2         WHEN A CUTE GIRL LEFT YOUR MESSAGE ON SEEN           0     0.170898  
3  Photographing something you want to show every...           1     0.579102  
4  HEY BABE CAN YOU MAKE ME A SANDWICH? Hey babe ...           0     0.424316  
Test Predictions Saved



/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


In [ ]:
ts=test_df

In [ ]:
test_df

/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,image_id,image_path,transcription,prediction,probability
0,15236,/content/drive/MyDrive/test_eng/15236.jpg,FACEBOOK SINGLES GROUPS BELIKE WHEN A NEW WOMA...,0,0.130371
1,15805,/content/drive/MyDrive/test_eng/15805.jpg,"SO, IF YOU'RE A FEMINIST HOW CAN YOU EAT DAIRY?",0,0.289795
2,16254,/content/drive/MyDrive/test_eng/16254.jpg,WHEN A CUTE GIRL LEFT YOUR MESSAGE ON SEEN,0,0.170898
3,16191,/content/drive/MyDrive/test_eng/16191.jpg,Photographing something you want to show every...,1,0.579102
4,15952,/content/drive/MyDrive/test_eng/15952.jpg,HEY BABE CAN YOU MAKE ME A SANDWICH? Hey babe ...,0,0.424316
...,...,...,...,...,...
995,15591,/content/drive/MyDrive/test_eng/15591.jpg,IT'S NOT YOUR FAULT You didn't design the dres...,1,0.845703
996,15049,/content/drive/MyDrive/test_eng/15049.jpg,THINK ABOUT HOW MUCH BETTER HER SKIN IS BREATH...,1,0.605957
997,15363,/content/drive/MyDrive/test_eng/15363.jpg,THE STEREOTYPES ARE TRUE F SHE DOES HAVE A TIG...,1,0.722656
998,15199,/content/drive/MyDrive/test_eng/15199.jpg,DRAWS NAKED PICTURES OF BLACK WOMEN 00 0000 GE...,0,0.479736


In [ ]:
ts = ts.drop(columns=['image_path'], inplace=True)

In [ ]:
ts

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score


merged = pred_df.merge(true_df, on="image_id")

y_pred = merged["prediction"]
y_true = merged["indian_labels"]

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

print("Accuracy :", acc)
print("Macro F1 :", macro_f1)

TypeError: Labels in y_true and y_pred should be of the same type. Got y_true=['misogyny' 'not-misogyny'] and y_pred=[0 1]. Make sure that the predictions provided by the classifier coincides with the true labels.

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

pred_df = pd.read_csv("/content/test_predictions (3).csv")
true_df = pd.read_csv("/content/test.csv")

true_df["indian_labels"] = true_df["indian_labels"].map({
    "not-misogyny": 0,
    "misogyny": 1
})

merged = pred_df.merge(true_df, on="image_id")

y_pred = merged["prediction"]
y_true = merged["indian_labels"]

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

print("Accuracy :", acc)
print("Macro F1 :", macro_f1)

Accuracy : 0.738
Macro F1 : 0.7170675236712972
